In [1]:
# %%
#
# Cell 1: Initial Setup
#
import pandas as pd
import numpy as np
import torch
import random
import os

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer,
    DataCollatorWithPadding
)

from datasets import Dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_recall_fscore_support,
    matthews_corrcoef, balanced_accuracy_score,
    cohen_kappa_score, jaccard_score, hamming_loss,
    confusion_matrix
)



/usr/local/lib/python3.10/dist-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/usr/local/lib/python3.10/dist-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
2025-06-28 19:27:32.313652: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-28 19:27:32.352381: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
202

In [2]:
# %%
#
# Cell 2: W&B and Hugging Face Login
#
import wandb
import optuna
import huggingface_hub

import os
os.environ["WANDB_PROJECT"] = "roberta_degendered"

# NOTE: Replace with your actual keys or use environment variables
# wandb.login(key="YOUR_WANDB_KEY")
# huggingface_hub.login(token="YOUR_HF_TOKEN")


In [3]:
# %%
#
# Cell 3: Model Configuration
#
model_name = "roberta-base"
model_cache_path = "../scratch/cache/roberta_degendered"



In [4]:
# %%
#
# Cell 4: Data Preparation
#
# Ensure 'data/combined_letters_degendered.csv' exists at the specified path
df = pd.read_csv("data/combined_letters_degendered.csv")[["full_text", "label"]].dropna()
df["label"] = df["label"].astype(int)

# Perform train-test split, stratifying by label to maintain class distribution
X_train, X_test, y_train, y_test = train_test_split(
    df["full_text"],
    df["label"],
    test_size=0.2,
    stratify=df["label"],
)

# The tokenizer is always loaded from the base model
tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=model_cache_path)



tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [5]:
# %%
#
# Cell 5: Tokenization
#
# This function prepares your text data for the model by converting it into token IDs
def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding=False,
        max_length=512
    )
    tokens["labels"] = example["label"]
    return tokens

# Convert pandas Series to Hugging Face Dataset objects, then map the tokenization function
train_dataset = Dataset.from_dict({"text": X_train.tolist(), "label": y_train.tolist()})
test_dataset = Dataset.from_dict({"text": X_test.tolist(), "label": y_test.tolist()})

tokenized_train = train_dataset.map(tokenize, batched=True).remove_columns(["text"])
tokenized_test = test_dataset.map(tokenize, batched=True).remove_columns(["text"])

# Data Collator: Dynamically pads input sequences to the longest sequence in the batch
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)



Map:   0%|          | 0/7189 [00:00<?, ? examples/s]

Map:   0%|          | 0/1798 [00:00<?, ? examples/s]

In [6]:
# %%
#
# Cell 6: Metrics Function
#
# Compute metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    preds = logits.argmax(-1)
    accuracy = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted", zero_division=0)

    print("Confusion Matrix:", confusion_matrix(labels, preds))

    return {
        "f1_score": f1,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "mcc": matthews_corrcoef(labels, preds),
        "balanced_accuracy": balanced_accuracy_score(labels, preds),
        "cohen_kappa": cohen_kappa_score(labels, preds),
        "jaccard": jaccard_score(labels, preds, average="macro"),
        "hamming_loss": hamming_loss(labels, preds)
    }



In [7]:
# %%
#
# Cell 7: Model Initialization for Hyperparameter Optimization
#
def model_init(trial=None):
    # Load the base pre-trained model (RoBERTa)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,
        id2label={0: "female", 1: "male"},
        label2id={"female": 0, "male": 1},
        cache_dir=model_cache_path,
        device_map="auto"
    )
    model.config.pad_token_id = tokenizer.pad_token_id
    return model



In [8]:
# %%
#
# Cell 8: Optuna Hyperparameter Space
#
def optuna_hp_space(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-4, log=True),
        "num_train_epochs": trial.suggest_int("num_train_epochs", 3, 10),
        "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size", [8, 16, 32]),
        "weight_decay": trial.suggest_float("weight_decay", 0.0, 0.03, step=0.01),
    }



In [9]:
# %%
#
# Cell 9: Trainer for Hyperparameter Optimization
#
training_args_for_hpo = TrainingArguments(
    output_dir="../scratch/hpo_results_roberta_degendered",
    per_device_eval_batch_size=32,
    fp16=True,
    save_strategy="no",
    logging_steps=50,
    report_to="wandb",
    remove_unused_columns=False,
    load_best_model_at_end=False,
    eval_strategy="epoch",
)

trainer = Trainer(
    model_init=model_init,
    args=training_args_for_hpo,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)



/tmp/ipykernel_2388438/1664402676.py:17: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
# %%
#
# Cell 10: Run Hyperparameter Search
#
best_run = trainer.hyperparameter_search(
    direction="maximize",
    backend="optuna",
    hp_space=optuna_hp_space,
    n_trials=20,
    compute_objective=lambda metrics: metrics["eval_f1_score"]
)

print("Best run details:")
print(best_run)

# Access the best hyperparameters found by Optuna
best_hps = best_run.hyperparameters
print("Best Hyperparameters Found:")
for hp, value in best_hps.items():
    print(f"  {hp}: {value}")

# W&B will provide a URL to the best run in its logs.
if hasattr(best_run, 'url'):
    print(f"Find the best run and explore all trials in W&B at: {best_run.url}")



[I 2025-06-28 19:27:53,910] A new study created in memory with name: no-name-830bb0cc-dcaa-41c0-a15b-5d7724b20168
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: mtwesley to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,F1 Score,Accuracy,Precision,Recall,Mcc,Balanced Accuracy,Cohen Kappa,Jaccard,Hamming Loss,Runtime,Samples Per Second,Steps Per Second
1,0.613500,0.619640,0.563707,0.690211,0.476392,0.690211,0.000000,0.500000,0.000000,0.345106,0.309789,1.465300,1227.035000,38.899000
2,0.609700,0.628535,0.563707,0.690211,0.476392,0.690211,0.000000,0.500000,0.000000,0.345106,0.309789,1.456400,1234.523000,39.137000
3,0.611000,0.609606,0.576882,0.694661,0.722375,0.694661,0.091176,0.508666,0.023698,0.356212,0.305339,1.457100,1233.939000,39.118000
4,0.603000,0.649571,0.630859,0.638487,0.625276,0.638487,0.123120,0.559009,0.122504,0.412640,0.361513,1.454700,1235.994000,39.183000
5,0.503800,0.688146,0.628120,0.632369,0.624548,0.632369,0.121837,0.559525,0.121638,0.410890,0.367631,1.459100,1232.272000,39.065000
6,0.414900,0.774290,0.635638,0.642380,0.630592,0.642380,0.135615,0.565293,0.135063,0.418086,0.357620,1.453700,1236.882000,39.211000
7,0.330200,0.911950,0.624113,0.621802,0.626666,0.621802,0.126917,0.564239,0.126847,0.409537,0.378198,1.452900,1237.528000,39.232000
8,0.287700,0.965773,0.631995,0.639600,0.626443,0.639600,0.125834,0.560310,0.125204,0.413867,0.360400,1.460500,1231.045000,39.026000


Confusion Matrix: [[   0  557]
 [   0 1241]]
Confusion Matrix: [[   0  557]
 [   0 1241]]
Confusion Matrix: [[  11  546]
 [   3 1238]]
Confusion Matrix: [[195 362]
 [288 953]]
Confusion Matrix: [[205 352]
 [309 932]]
Confusion Matrix: [[202 355]
 [288 953]]
Confusion Matrix: [[230 327]
 [353 888]]
Confusion Matrix: [[196 361]
 [287 954]]


[I 2025-06-28 19:30:11,517] Trial 0 finished with value: 0.6319946804984954 and parameters: {'learning_rate': 2.249527253601209e-05, 'num_train_epochs': 8, 'per_device_train_batch_size': 32, 'weight_decay': 0.0}. Best is trial 0 with value: 0.6319946804984954.
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


eval/accuracy,███▃▂▃▁▃
eval/balanced_accuracy,▁▁▂▇▇██▇
eval/cohen_kappa,▁▁▂▇▇██▇
eval/f1_score,▁▁▂█▇█▇█
eval/hamming_loss,▁▁▁▆▇▆█▆
eval/jaccard,▁▁▂▇▇█▇█
eval/loss,▁▁▁▂▃▄▇█
eval/mcc,▁▁▆▇▇██▇
eval/precision,▁▁█▅▅▅▅▅
eval/recall,███▃▂▃▁▃
eval/runtime,█▃▃▂▅▁▁▅


...

eval/accuracy,▁▁▁
eval/balanced_accuracy,▁▁▁
eval/cohen_kappa,▁▁▁
eval/f1_score,▁▁▁
eval/hamming_loss,▁▁▁
eval/jaccard,▁▁▁
eval/loss,█▅▁
eval/mcc,▁▁▁
eval/precision,▁▁▁
eval/recall,▁▁▁
eval/runtime,█▁▄


Epoch,Training Loss,Validation Loss,F1 Score,Accuracy,Precision,Recall,Mcc,Balanced Accuracy,Cohen Kappa,Jaccard,Hamming Loss,Runtime,Samples Per Second,Steps Per Second
1,0.618100,0.620748,0.563707,0.690211,0.476392,0.690211,0.000000,0.500000,0.000000,0.345106,0.309789,1.450800,1239.350000,39.290000
2,0.645200,0.623927,0.563707,0.690211,0.476392,0.690211,0.000000,0.500000,0.000000,0.345106,0.309789,1.452000,1238.287000,39.256000
3,0.622200,0.620612,0.563707,0.690211,0.476392,0.690211,0.000000,0.500000,0.000000,0.345106,0.309789,1.450300,1239.761000,39.303000
4,0.650600,0.621303,0.563707,0.690211,0.476392,0.690211,0.000000,0.500000,0.000000,0.345106,0.309789,1.451500,1238.736000,39.270000
5,0.678600,0.618864,0.563707,0.690211,0.476392,0.690211,0.000000,0.500000,0.000000,0.345106,0.309789,1.452500,1237.833000,39.242000
6,0.622100,0.618925,0.563707,0.690211,0.476392,0.690211,0.000000,0.500000,0.000000,0.345106,0.309789,1.464200,1227.978000,38.929000


Confusion Matrix: [[   0  557]
 [   0 1241]]
Confusion Matrix: [[   0  557]
 [   0 1241]]
Confusion Matrix: [[   0  557]
 [   0 1241]]
Confusion Matrix: [[   0  557]
 [   0 1241]]
Confusion Matrix: [[   0  557]
 [   0 1241]]
Confusion Matrix: [[   0  557]
 [   0 1241]]


[I 2025-06-28 19:54:54,276] Trial 19 finished with value: 0.5637066668716401 and parameters: {'learning_rate': 3.941786868307671e-05, 'num_train_epochs': 6, 'per_device_train_batch_size': 8, 'weight_decay': 0.01}. Best is trial 4 with value: 0.6620333730401097.


Best run details:
BestRun(run_id='4', objective=0.6620333730401097, hyperparameters={'learning_rate': 2.032248008048514e-05, 'num_train_epochs': 6, 'per_device_train_batch_size': 16, 'weight_decay': 0.0}, run_summary=None)
Best Hyperparameters Found:
  learning_rate: 2.032248008048514e-05
  num_train_epochs: 6
  per_device_train_batch_size: 16
  weight_decay: 0.0


In [11]:
# %%
#
# Cell 11: Final Model Retraining
#
# Re-training the model with the best hyperparameters
best_hps = best_run.hyperparameters

# Re-initialize the base model
final_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label={0: "female", 1: "male"},
    label2id={"female": 0, "male": 1},
    cache_dir=model_cache_path,
    device_map="auto"
)

final_model.config.pad_token_id = tokenizer.pad_token_id

# Define TrainingArguments for the final training run
final_model_output_dir = "../scratch/final_roberta_degendered_model"
final_training_args = TrainingArguments(
    output_dir=final_model_output_dir,
    per_device_train_batch_size=best_hps["per_device_train_batch_size"],
    per_device_eval_batch_size=best_hps["per_device_train_batch_size"],
    num_train_epochs=best_hps["num_train_epochs"],
    learning_rate=best_hps["learning_rate"],
    weight_decay=best_hps["weight_decay"],
    fp16=True,
    save_strategy="epoch",
    logging_steps=50,
    report_to="wandb",
    remove_unused_columns=False,
    load_best_model_at_end=True,
    metric_for_best_model="f1_score",
    eval_strategy="epoch",
    save_total_limit=1,
    run_name="final_roberta_model_training"
)

# Initialize the Trainer for final training
final_trainer = Trainer(
    model=final_model,
    args=final_training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# Train the final model
final_trainer.train()

# Evaluate the final (best) model
eval_results = final_trainer.evaluate()
print("Final Model Evaluation Results:")
print(eval_results)

# Save the final trained model and tokenizer
final_trainer.save_model(final_model_output_dir)
tokenizer.save_pretrained(final_model_output_dir)

print(f"Final model and tokenizer saved to: {final_model_output_dir}")


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_2388438/1999929638.py:42: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  final_trainer = Trainer(


Epoch,Training Loss,Validation Loss,F1 Score,Accuracy,Precision,Recall,Mcc,Balanced Accuracy,Cohen Kappa,Jaccard,Hamming Loss,Runtime,Samples Per Second,Steps Per Second
1,0.614400,0.622301,0.563707,0.690211,0.476392,0.690211,0.000000,0.500000,0.000000,0.345106,0.309789,1.609100,1117.412000,70.227000
2,0.606900,0.637908,0.563707,0.690211,0.476392,0.690211,0.000000,0.500000,0.000000,0.345106,0.309789,1.602200,1122.201000,70.528000
3,0.587000,0.615547,0.626001,0.677419,0.627868,0.677419,0.109937,0.538726,0.093115,0.401152,0.322581,1.603200,1121.487000,70.483000
4,0.566800,0.663440,0.626985,0.622914,0.631883,0.622914,0.138930,0.570982,0.138671,0.413655,0.377086,1.602800,1121.754000,70.500000
5,0.447100,0.719407,0.648143,0.660178,0.641413,0.660178,0.159430,0.574228,0.157412,0.430036,0.339822,1.604300,1120.740000,70.436000
6,0.346700,0.800677,0.639229,0.644605,0.635002,0.644605,0.146086,0.570863,0.145690,0.422478,0.355395,1.604800,1120.399000,70.414000


Confusion Matrix: [[   0  557]
 [   0 1241]]
Confusion Matrix: [[   0  557]
 [   0 1241]]
Confusion Matrix: [[  97  460]
 [ 120 1121]]
Confusion Matrix: [[242 315]
 [363 878]]
Confusion Matrix: [[194 363]
 [248 993]]
Confusion Matrix: [[210 347]
 [292 949]]


Confusion Matrix: [[194 363]
 [248 993]]
Final Model Evaluation Results:
{'eval_loss': 0.7194070816040039, 'eval_f1_score': 0.6481426260785776, 'eval_accuracy': 0.6601779755283649, 'eval_precision': 0.6414129651569858, 'eval_recall': 0.6601779755283649, 'eval_mcc': 0.15942964473226665, 'eval_balanced_accuracy': 0.5742277974124649, 'eval_cohen_kappa': 0.15741202349414862, 'eval_jaccard': 0.4300355477765214, 'eval_hamming_loss': 0.3398220244716352, 'eval_runtime': 1.6146, 'eval_samples_per_second': 1113.558, 'eval_steps_per_second': 69.984, 'epoch': 6.0}
Final model and tokenizer saved to: ../scratch/final_roberta_degendered_model
